In [ ]:
from trl.trainer.sft_trainer import SFTTrainer
from trl.trainer.sft_config import SFTConfig

In [23]:
# 1、处理数据，处理成SFTTrainer能够接收的数据类型
from datasets import load_dataset
dataset = load_dataset("json",data_files={"train":r"data\keywords_data_train.jsonl","test":r"data\keywords_data_test.jsonl"})

In [24]:
dataset

DatasetDict({
    train: Dataset({
        features: ['conversation_id', 'category', 'conversation', 'dataset'],
        num_rows: 49500
    })
    test: Dataset({
        features: ['conversation_id', 'category', 'conversation', 'dataset'],
        num_rows: 500
    })
})

In [25]:
from typing import List,Dict
def convert_to_messages_format(examples:Dict[str,List]):
    conversations:List[List[Dict]] = examples["conversation"]
    result = []
    for conversation in conversations:
        # 遍历一个batch当中的一条数据
        all_messages:Dict = conversation[0]
        new_message_list = []
        new_message_list.append({"role":"user","content":all_messages["human"]})
        new_message_list.append({"role":"assistant","content":all_messages["assistant"]})
        result.append(new_message_list)
    
    return {"messages":result}


mapped_dataset = dataset.map(convert_to_messages_format,batched=True,remove_columns=['conversation_id', 'category', 'conversation', 'dataset'])

In [26]:
mapped_dataset["train"][0]

{'messages': [{'content': '高氟铍矿石在熔炼过程中配入氢氧化铝来脱除其中的氟.结果表明,在配入5％Na2CO3、9.3％Al(OH)3、1400～1500℃熔炼20 min的情况下,BeO回收率达到96％以上,脱氟效果良好(铍玻璃F/BeO能控制在15％以内).为高氟铍矿石的工业应用探索出新的冶炼途径.\n找出上文中的关键词',
   'role': 'user'},
  {'content': '高氟铍矿;配料;熔炼;回收率;脱氟率', 'role': 'assistant'}]}

In [ ]:
# 2、构造SFTConfig实例
import os
os.environ["TENSORBOARD_LOGGING_DIR"] = "logs/Qwen3-0.6B-TRL-SFT"
config = SFTConfig(
    output_dir="./finetuned/Qwen3-0.6B-TRL-SFT",
    per_device_train_batch_size=3,
    gradient_accumulation_steps=4,
    #num_train_epochs=1,
    max_steps=1000,
    learning_rate=2e-5,
    lr_scheduler_type="cosine",
    warmup_ratio=0.1,
    # warmup_steps = ,
    bf16=True,
    gradient_checkpointing=True,
    logging_strategy="steps",
    logging_steps=100,
    save_steps=100,
    save_strategy="steps",
    eval_strategy="steps",
    eval_steps=100,
    max_length=2500,
    assistant_only_loss=True,
    chat_template_path="./chat_template.jinja",
    report_to=["tensorboard"]
)


warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
# 3、加载模型和tokenizer
model = AutoModelForCausalLM.from_pretrained(r"model/Qwen3-0.6B/")
tokenizer = AutoTokenizer.from_pretrained(r"model/Qwen3-0.6B/")

# 4、构造SFTTrainer实例
trainer = SFTTrainer(
    model=model,
    processing_class=tokenizer,
    args=config,
    train_dataset=mapped_dataset["train"],
    eval_dataset=mapped_dataset["test"]
)

Loading weights: 100%|██████████| 311/311 [00:00<00:00, 1168.89it/s, Materializing param=model.norm.weight]                              
The tied weights mapping and config for this model specifies to tie model.embed_tokens.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


In [31]:

# 5、测试，查看一下当前的数据
train_dataloader = trainer.get_train_dataloader()
for batch_data in train_dataloader:
    print(tokenizer.decode(batch_data["input_ids"])[0])
    break

<|im_start|>user
关键词抽取：
以手动换挡机构疲劳寿命试验为目的,构建了一种模拟驾驶员进行选档、换挡操作的试验平台,该平台集成二自由度运动滑台与气动加载装置为一体形成换挡运动加载机构.以该机构为研究对象,通过建立换挡与选挡的运动轨迹模型,分析换挡运动加载机构位移输出与运动轨迹之间的关系,通过分析换挡机构操纵杆受力情况,分别对换挡动作和选档动作进行力学分析,并得出加载力的计算方法.最后结合电气控制技术与气动控制技术,对系统进行了试验,结果表明,系统具有可行性与正确性.<|im_end|>
<|im_start|>assistant

<think>

</think>

换挡机构;试验平台;疲劳性能<|im_end|>




In [ ]:

# 6、调用SFTTrainer的train进行训练
trainer.train()

# 7、调用SFTTrainer实例去保存模型参数和Tokenizer
trainer.save_model("./finetuned/Qwen3-0.6B-TRL-SFT")


In [ ]:
from transformers.integrations.integration_utils import TensorBoardCallback